In [7]:
import torch 
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from pathlib import Path

In [8]:
#Check the no of images in each class 
dataset_path = Path("../data/RealWaste")

for category in sorted(dataset_path.iterdir()):
    if category.is_dir():
        image_count = len(list(category.glob("*")))
        print(f"{category.name}: {image_count} images")

Cardboard: 461 images
Food Organics: 411 images
Glass: 420 images
Metal: 790 images
Miscellaneous Trash: 495 images
Paper: 500 images
Plastic: 921 images
Textile Trash: 318 images
Vegetation: 436 images


In [9]:
#Split the dataset into training and testing sets
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# Path to the dataset
dataset_path = Path("../data/RealWaste")

# Collect image paths and their class labels
data = []

for class_folder in sorted(dataset_path.iterdir()):
    if class_folder.is_dir():
        for image_path in class_folder.iterdir():
            if image_path.is_file():
                data.append({
                    "image_path": str(image_path),
                    "label": class_folder.name
                })

df = pd.DataFrame(data)


train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)
print("\nTraining set size:", len(train_df))
print("Temporary set size:", len(temp_df))

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


Training set size: 3326
Temporary set size: 1426
Train: 3326
Validation: 713
Test: 713


In [10]:
print("TRAIN")
print(train_df["label"].value_counts().sort_index())

print("\nVALIDATION")
print(val_df["label"].value_counts().sort_index())

print("\nTEST")
print(test_df["label"].value_counts().sort_index())

TRAIN
label
Cardboard              323
Food Organics          288
Glass                  294
Metal                  553
Miscellaneous Trash    346
Paper                  350
Plastic                645
Textile Trash          222
Vegetation             305
Name: count, dtype: int64

VALIDATION
label
Cardboard               69
Food Organics           61
Glass                   63
Metal                  119
Miscellaneous Trash     74
Paper                   75
Plastic                138
Textile Trash           48
Vegetation              66
Name: count, dtype: int64

TEST
label
Cardboard               69
Food Organics           62
Glass                   63
Metal                  118
Miscellaneous Trash     75
Paper                   75
Plastic                138
Textile Trash           48
Vegetation              65
Name: count, dtype: int64


In [11]:
import torch
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

# Load the pretrained MobileNetV2 weights
weights = MobileNet_V2_Weights.DEFAULT

# Create the pretrained model
model = mobilenet_v2(weights=weights)

print(model)

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
  

In [12]:
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [13]:
#Replace the 1000 class classifier with a new classifier for 9 classes
num_classes = 9

model.classifier[1] = torch.nn.Linear(
    model.classifier[1].in_features,
    num_classes
)

print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=9, bias=True)
)


In [14]:
from torchvision import transforms

# ImageNet normalization used by the pretrained MobileNetV2
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# Transformations for training images (Data augmenatation allowed as we dont want the model to overfit)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), #Convert the image to a tensor
    normalize
])

# Transformations for validation/test images
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])

print("Transforms created successfully.")

Transforms created successfully.


In [15]:
#create class labels 
class_names = sorted(train_df["label"].unique())

print(class_names)

['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']


In [16]:
from torch.utils.data import Dataset
from PIL import Image


class WasteDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        # Get image path and label from the DataFrame
        image_path = self.dataframe.iloc[index]["image_path"]
        label_name = self.dataframe.iloc[index]["label"]

        # Open the image
        image = Image.open(image_path).convert("RGB")

        # Convert the class name into a number
        label = class_names.index(label_name)

        # Apply the appropriate transformations
        if self.transform:
            image = self.transform(image)

        return image, label

In [17]:
train_dataset = WasteDataset(
    train_df,
    transform=train_transform
)

print("Number of training images:", len(train_dataset))

Number of training images: 3326


In [18]:
image, label = train_dataset[0]

print("Image shape:", image.shape)
print("Label:", label)
print("Class:", class_names[label])

Image shape: torch.Size([3, 224, 224])
Label: 5
Class: Paper


In [19]:
val_dataset = WasteDataset(
    val_df,
    transform=val_test_transform
)

test_dataset = WasteDataset(
    test_df,
    transform=val_test_transform
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

Training images: 3326
Validation images: 713
Test images: 713


In [20]:
#Create data loaders for training, validation, and testing

from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)



In [21]:
images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch labels shape:", labels.shape)

Batch image shape: torch.Size([32, 3, 224, 224])
Batch labels shape: torch.Size([32])


In [22]:
#Keep the pretrained weights frozen 
# Freeze the pretrained MobileNetV2 backbone
for parameter in model.features.parameters():
    parameter.requires_grad = False

print("MobileNetV2 backbone frozen.")

MobileNetV2 backbone frozen.


In [23]:
criterion = torch.nn.CrossEntropyLoss()

print(criterion)
optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=0.001
)

print(optimizer)

CrossEntropyLoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [24]:
device = torch.device("cpu")

model = model.to(device)

print("Using device:", device)

Using device: cpu


In [25]:
images = images.to(device)
labels = labels.to(device)

outputs = model(images)

print("Output shape:", outputs.shape)

Output shape: torch.Size([32, 9])


In [26]:
loss = criterion(outputs, labels)

print("Loss:", loss.item())

Loss: 2.221930980682373


In [28]:
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

print("Images:", images.shape)
print("Labels:", labels.shape)

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])


In [29]:
outputs = model(images)

print("Output shape:", outputs.shape)

Output shape: torch.Size([32, 9])


In [30]:
loss = criterion(outputs, labels)

print("Loss:", loss.item())

Loss: 2.2228715419769287


In [31]:
optimizer.zero_grad()

In [32]:
loss.backward()

print("Backpropagation completed.")

Backpropagation completed.


In [33]:
optimizer.step()

print("Weights updated.")

Weights updated.


In [35]:
model.train()

running_loss = 0.0
correct = 0
total = 0

for images, labels in train_loader:

    images = images.to(device)
    labels = labels.to(device)

    # Forward pass
    outputs = model(images)

    # Calculate loss
    loss = criterion(outputs, labels)

    # Clear old gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    # Keep track of loss
    running_loss += loss.item()

    # Get predicted class
    _, predicted = torch.max(outputs, 1)

    # Count correct predictions
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

# Calculate average loss and accuracy
average_loss = running_loss / len(train_loader)
accuracy = 100 * correct / total

print("Training Loss:", average_loss)
print("Training Accuracy:", accuracy, "%")

Training Loss: 0.9385701068318807
Training Accuracy: 71.4371617558629 %


In [36]:
num_epochs = 4

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Keep track of loss
        running_loss += loss.item()

        # Get predicted class
        _, predicted = torch.max(outputs, 1)

        # Count correct predictions
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    average_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total

    print(
        f"Epoch {epoch + 2}/5 "
        f"- Loss: {average_loss:.4f} "
        f"- Accuracy: {accuracy:.2f}%"
    )

Epoch 2/5 - Loss: 0.7946 - Accuracy: 75.26%
Epoch 3/5 - Loss: 0.7165 - Accuracy: 76.76%
Epoch 4/5 - Loss: 0.6520 - Accuracy: 79.13%
Epoch 5/5 - Loss: 0.6190 - Accuracy: 79.89%


In [37]:
model.eval()

val_loss = 0.0
correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Keep track of loss
        val_loss += loss.item()

        # Get predicted class
        _, predicted = torch.max(outputs, 1)

        # Count correct predictions
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

average_val_loss = val_loss / len(val_loader)
val_accuracy = 100 * correct / total

print("Validation Loss:", average_val_loss)
print("Validation Accuracy:", val_accuracy, "%")

Validation Loss: 0.6246757688729659
Validation Accuracy: 77.13884992987377 %


In [38]:
#5 more epochs of training
num_epochs = 5

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Track loss
        running_loss += loss.item()

        # Track accuracy
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    model.eval()

    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)

            # Calculate loss
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            # Get predictions
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * correct / total

    print(
        f"Epoch {epoch + 6}/10 "
        f"- Train Loss: {train_loss:.4f} "
        f"- Train Acc: {train_accuracy:.2f}% "
        f"- Val Loss: {val_loss:.4f} "
        f"- Val Acc: {val_accuracy:.2f}%"
    )

Epoch 6/10 - Train Loss: 0.5704 - Train Acc: 81.99% - Val Loss: 0.5844 - Val Acc: 79.66%
Epoch 7/10 - Train Loss: 0.5439 - Train Acc: 81.90% - Val Loss: 0.5710 - Val Acc: 79.38%
Epoch 8/10 - Train Loss: 0.5255 - Train Acc: 83.58% - Val Loss: 0.5442 - Val Acc: 81.35%
Epoch 9/10 - Train Loss: 0.5144 - Train Acc: 83.40% - Val Loss: 0.5364 - Val Acc: 81.35%
Epoch 10/10 - Train Loss: 0.4795 - Train Acc: 85.48% - Val Loss: 0.5321 - Val Acc: 80.79%


In [39]:
for i, layer in enumerate(model.features):
    print(i, layer)

0 Conv2dNormActivation(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU6(inplace=True)
)
1 InvertedResidual(
  (conv): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
)
2 InvertedResidual(
  (conv): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU6(inplace=Tru

In [40]:
for parameter in model.features[14:].parameters():
    parameter.requires_grad = True

In [41]:
backbone_trainable = sum(
    p.requires_grad for p in model.features.parameters()
)

classifier_trainable = sum(
    p.requires_grad for p in model.classifier.parameters()
)

print("Trainable backbone parameters:", backbone_trainable)
print("Trainable classifier parameters:", classifier_trainable)

Trainable backbone parameters: 39
Trainable classifier parameters: 2


In [42]:
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

In [43]:
num_epochs = 5

for epoch in range(num_epochs):

    # Training
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * correct / total

    print(
        f"Fine-tuning Epoch {epoch + 1}/5 "
        f"- Train Loss: {train_loss:.4f} "
        f"- Train Acc: {train_accuracy:.2f}% "
        f"- Val Loss: {val_loss:.4f} "
        f"- Val Acc: {val_accuracy:.2f}%"
    )

Fine-tuning Epoch 1/5 - Train Loss: 0.4208 - Train Acc: 85.75% - Val Loss: 0.4221 - Val Acc: 84.99%
Fine-tuning Epoch 2/5 - Train Loss: 0.2711 - Train Acc: 91.67% - Val Loss: 0.3603 - Val Acc: 87.52%
Fine-tuning Epoch 3/5 - Train Loss: 0.1962 - Train Acc: 94.47% - Val Loss: 0.3347 - Val Acc: 87.94%
Fine-tuning Epoch 4/5 - Train Loss: 0.1482 - Train Acc: 96.33% - Val Loss: 0.3213 - Val Acc: 88.92%
Fine-tuning Epoch 5/5 - Train Loss: 0.1072 - Train Acc: 97.32% - Val Loss: 0.3065 - Val Acc: 89.20%


In [44]:
torch.save(model.state_dict(), "mobilenetv2_realwaste_best.pth")



In [45]:
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Make predictions
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        # Get predicted class
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_loss = test_loss / len(test_loader)
test_accuracy = 100 * correct / total

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy, "%")

Test Loss: 0.3444255590438843
Test Accuracy: 88.49929873772791 %


In [46]:
model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Make predictions
        outputs = model(images)

        # Get predicted class
        _, predicted = torch.max(outputs, 1)

        # Store actual and predicted labels
        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

print("Predictions collected:", len(all_predictions))
print("Actual labels collected:", len(all_labels))

Predictions collected: 713
Actual labels collected: 713


In [47]:
from sklearn.metrics import classification_report

report = classification_report(
    all_labels,
    all_predictions,
    target_names=class_names
)

print(report)

                     precision    recall  f1-score   support

          Cardboard       0.94      0.93      0.93        69
      Food Organics       0.93      0.92      0.93        62
              Glass       0.85      0.89      0.87        63
              Metal       0.85      0.89      0.87       118
Miscellaneous Trash       0.77      0.79      0.78        75
              Paper       0.99      0.88      0.93        75
            Plastic       0.87      0.84      0.86       138
      Textile Trash       0.88      0.96      0.92        48
         Vegetation       0.95      0.95      0.95        65

           accuracy                           0.88       713
          macro avg       0.89      0.89      0.89       713
       weighted avg       0.89      0.88      0.89       713



In [49]:
#create a confusion matrix
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(all_labels, all_predictions)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

print(cm_df)

                     Cardboard  Food Organics  Glass  Metal  \
Cardboard                   64              0      2      2   
Food Organics                0             57      0      0   
Glass                        1              0     56      3   
Metal                        0              0      1    105   
Miscellaneous Trash          0              2      1      1   
Paper                        2              1      1      2   
Plastic                      1              1      5     11   
Textile Trash                0              0      0      0   
Vegetation                   0              0      0      0   

                     Miscellaneous Trash  Paper  Plastic  Textile Trash  \
Cardboard                              0      1        0              0   
Food Organics                          1      0        2              0   
Glass                                  2      0        1              0   
Metal                                  5      0        7             